# 03 · Visualización

**Duración estimada:** 40 minutos

## Propósito

Tenemos resultados. Falta lo más difícil: **mostrarlos sin mentir.**

Una imagen convence más rápido que una tabla, y por eso mismo es más fácil que
convenza de algo que los datos no dicen. Este cuaderno se organiza alrededor de
una sola pregunta, que vamos a repetir hasta el cansancio:

> **¿La imagen sostiene lo que estamos afirmando?**

## Qué consume

- `datos/corpus_limpio.jsonl`

## Celda de reanudación

In [ ]:
# pandas: para las tablas.
import pandas as pd

# matplotlib: la librería de gráficos.
import matplotlib.pyplot as plt

# Counter: para contar.
from collections import Counter

corpus = pd.read_json("datos/corpus_limpio.jsonl", lines=True)

archivo = open("datos/stopwords_es.txt", encoding="utf-8")
palabras_vacias = archivo.read().split()
archivo.close()

anios = sorted(corpus["anio"].unique())

print("párrafos:", f"{len(corpus):,}")

In [ ]:
# Recalculamos lo del cuaderno 01: palabras de cada año y tamaño de cada año.
palabras_por_anio = {}
total_por_anio = {}

for anio in anios:
    parrafos_del_anio = corpus[corpus["anio"] == anio]
    texto_del_anio = " ".join(parrafos_del_anio["texto_limpio"])
    palabras = texto_del_anio.split()
    palabras_por_anio[anio] = palabras
    total_por_anio[anio] = len(palabras)

for anio in anios:
    print(anio, "->", f"{total_por_anio[anio]:,}", "palabras")

## 1. Configurar los gráficos una sola vez

matplotlib viene con valores por omisión que no son buenos: rejillas gruesas,
marcos por los cuatro lados, letra chica.

En vez de repetir la misma configuración en cada gráfico, la fijamos una vez acá
y todos los gráficos del cuaderno la heredan.

In [ ]:
# --- Los colores de la unidad ---

# La superficie: un blanco apenas cálido, menos duro que el blanco puro.
SUPERFICIE = "#fcfcfb"

# La tinta: un gris muy oscuro con una gota de violeta, en vez de negro puro.
TINTA = "#3d3a4a"

# Dos colores para los datos: uno principal y uno de acento.
VIOLETA = "#6437b5"
ROSA = "#d1568c"

# Y un neutro, para lo que queremos que se lea pero no se destaque.
NEUTRO = "#8d8a99"

# --- Una rampa ordenada, de lo más frecuente a lo menos ---
# Va de un índigo profundo a un rosa. Cada paso es más claro que el anterior.
RAMPA = ["#2b1b6b", "#5b2ea6", "#8f3bbf", "#c4419f", "#e0557b"]

print("paleta lista")

In [ ]:
# --- La configuración de matplotlib, una sola vez ---

# Tamaño por omisión de las figuras, en pulgadas.
plt.rcParams["figure.figsize"] = (8, 4)

# El fondo, en los dos lugares donde matplotlib lo pinta.
plt.rcParams["figure.facecolor"] = SUPERFICIE
plt.rcParams["axes.facecolor"] = SUPERFICIE

# Sacamos el marco de arriba y de la derecha: no aportan información.
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.edgecolor"] = "#c9c6d0"

# Rejilla fina y clarita, siempre por detrás de los datos.
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.color"] = "#d8d5de"
plt.rcParams["grid.linewidth"] = 0.7
plt.rcParams["axes.axisbelow"] = True

# Todo el texto en la misma tinta.
plt.rcParams["font.size"] = 10
plt.rcParams["text.color"] = TINTA
plt.rcParams["axes.labelcolor"] = TINTA
plt.rcParams["axes.titlecolor"] = TINTA
plt.rcParams["xtick.color"] = TINTA
plt.rcParams["ytick.color"] = TINTA

print("configuración lista")

### Por qué estos colores y no otros

Tres decisiones, y ninguna es de gusto.

**El fondo no es blanco puro.** `#fcfcfb` es un blanco apenas cálido. Doce
paneles de blanco puro cansan la vista; este no.

**Los datos usan dos colores, no ocho.** Cada color que agregás tiene que estar
diciendo algo. Un gráfico con ocho colores donde el color no significa nada se
lee peor que uno con dos.

**Y esos dos se pueden distinguir con daltonismo.** Entre el 5 y el 8% de los
varones no distingue bien rojo de verde. Un violeta y un rosa se separan bien
para casi todo el mundo; un rojo y un verde, no. Se puede verificar: hay
calculadoras de contraste y simuladores de daltonismo en línea, y conviene pasar
por ahí antes de dar una paleta por buena.

**La rampa es aparte y tiene otro trabajo.** Los cinco pasos de `RAMPA` van de un
índigo profundo a un rosa, cada uno más claro que el anterior. No es una paleta
de categorías: es una escala. Sirve para que el color diga **cuánto**.

In [ ]:
# --- La tipografía ---
# La que trae matplotlib por omisión funciona, pero es pesada.
# Buscamos una más liviana; si no está instalada, seguimos con la que haya.
from matplotlib import font_manager

RUTA_FUENTE = None
for nombre_de_fuente in ["Carlito", "Liberation Sans", "DejaVu Sans"]:
    try:
        RUTA_FUENTE = font_manager.findfont(nombre_de_fuente, fallback_to_default=False)
        break
    except Exception:
        continue

print("tipografía:", RUTA_FUENTE)

## 2. La nube de palabras

Es lo que todo el mundo espera de un análisis de texto. Hagámosla, con el corpus
entero.

In [ ]:
# wordcloud: la librería de nubes de palabras.
# Si no la tenés:  uv pip install wordcloud
from wordcloud import WordCloud

todo_el_corpus = " ".join(corpus["texto_limpio"])

nube = WordCloud(
    width=900,
    height=450,
    background_color=SUPERFICIE,
    colormap="Purples",
    stopwords=set(palabras_vacias),
    max_words=80,
    random_state=42,
    font_path=RUTA_FUENTE,
)
nube.generate(todo_el_corpus)

plt.figure(figsize=(10, 5))
plt.imshow(nube, interpolation="bilinear")
plt.axis("off")
plt.grid(False)
plt.show()

Es linda. Ahora las preguntas incómodas.

**¿Qué afirmación sostiene esta imagen?**

Ninguna en particular. Muestra que ciertas palabras son frecuentes en un corpus
de doce años. No dice cuánto, ni comparado con qué, ni si eso cambió.

**¿Qué número podés verificar acá?**

Ninguno. El tamaño es proporcional a la frecuencia, pero no hay escala ni eje.
No podés decir cuántas veces aparece `casa`, ni si aparece el doble que `mundo`.

**¿Qué decide la posición y el color?**

El azar. `random_state=42` fija ese azar para que la imagen salga siempre igual;
cambiá el número y la nube se reordena entera sin que ningún dato haya
cambiado.

In [ ]:
figura, paneles = plt.subplots(1, 2, figsize=(11, 4))

semillas = [42, 7]

for posicion in range(2):
    semilla = semillas[posicion]
    panel = paneles[posicion]
    otra_nube = WordCloud(
        width=600, height=300, background_color=SUPERFICIE, colormap="Purples",
        stopwords=set(palabras_vacias), max_words=40, random_state=semilla,
        font_path=RUTA_FUENTE,
    )
    otra_nube.generate(todo_el_corpus)
    panel.imshow(otra_nube, interpolation="bilinear")
    panel.axis("off")
    panel.grid(False)
    panel.set_title("random_state = " + str(semilla))

plt.show()

Los mismos datos, dos imágenes distintas.

Es un **título-etiqueta**: dice "de esto habla el corpus". No afirma nada.

**Pero eso no es culpa de la nube de palabras. Es culpa de haber hecho una sola.**

### Doce nubes en vez de una

Volvamos a la pregunta que nos trajo hasta acá: **¿cambia lo que escribe Casciari
a lo largo de doce años?**

Hagamos una nube **por año**, todas con la misma configuración, ordenadas de 2004
a 2015, y con **diez palabras cada una**.

Antes de dibujarlas, una decisión de diseño que también es de análisis.

En las nubes de arriba **el color lo elige el azar**: no dice nada, y encima las
palabras más claras casi no se leen. Vamos a usarlo para otra cosa: que el color
diga **el puesto de cada palabra**. La más frecuente en el índigo más oscuro de
la rampa, la menos frecuente en el rosa.

Así el color refuerza lo que ya dice el tamaño, en vez de agregar ruido.

In [ ]:
# Este diccionario se vuelve a llenar en cada vuelta del bucle,
# con el puesto de cada palabra dentro de su año.
puestos = {}


def color_por_puesto(word, font_size, position, orientation, font_path, random_state):
    # WordCloud llama a esta función una vez por palabra, con todos esos datos.
    # Nosotros solo usamos la palabra: buscamos su puesto y elegimos un paso
    # de la rampa. Puesto 0 (la más frecuente) -> el color más oscuro.
    puesto = puestos[word]
    indice = puesto * len(RAMPA) // len(puestos)
    return RAMPA[indice]


print("listo")

In [ ]:
figura, paneles = plt.subplots(4, 3, figsize=(13.5, 14))
figura.patch.set_facecolor(SUPERFICIE)

# .flatten() convierte la grilla de 4x3 en una lista de 12 paneles,
# para poder recorrerla con un solo bucle.
lista_de_paneles = paneles.flatten()

for posicion in range(len(anios)):
    anio = anios[posicion]
    texto_del_anio = " ".join(palabras_por_anio[anio])

    nube_del_anio = WordCloud(
        width=640,
        height=420,
        background_color=SUPERFICIE,
        stopwords=set(palabras_vacias),
        max_words=10,
        max_font_size=100,
        min_font_size=26,
        random_state=42,
        prefer_horizontal=1.0,
        relative_scaling=0.75,
        margin=12,
        font_path=RUTA_FUENTE,
    )
    nube_del_anio.generate(texto_del_anio)

    # Anotamos el puesto de cada palabra y recoloreamos.
    puestos.clear()
    puesto = 0
    for palabra in nube_del_anio.words_:
        puestos[palabra] = puesto
        puesto = puesto + 1
    nube_del_anio.recolor(color_func=color_por_puesto)

    panel = lista_de_paneles[posicion]
    panel.imshow(nube_del_anio, interpolation="bilinear")
    panel.axis("off")
    panel.grid(False)
    panel.set_title(str(anio), fontsize=14, pad=10)

figura.suptitle("Las diez palabras más frecuentes de cada año", fontsize=17, y=0.998)
figura.tight_layout(h_pad=2.2, w_pad=1.2)
plt.show()

**Ahora sí hay una afirmación, y salta a la vista sin hacer una sola cuenta:
2010, 2011 y 2012 son distintos de todos los demás.**

En esos tres paneles aparecen `revista`, `orsai`, `chiri`, `lectores`,
`librero`. En los otros nueve no hay ni rastro de eso.

Es exactamente el corte que encontramos en el cuaderno 02 con el cálculo de
palabras características —los años en que Casciari funda Editorial Orsai—, pero
acá se ve de un vistazo.

### Por qué esta imagen sí sostiene una afirmación

Lo que cambió no es la herramienta: siguen siendo nubes de palabras. Cambiaron
tres cosas.

**1. Son comparables.** Misma configuración, mismo tamaño, misma paleta, misma
semilla. Si cada panel tuviera colores y tamaños distintos, no se podrían
comparar.

**2. Están ordenadas.** De 2004 a 2015, leyendo como se lee una página. El orden
es un dato: es la variable que estamos estudiando.

**3. Están reducidas a diez palabras.** Esto es lo que más importa. Con ochenta
palabras cada panel es una mancha; con diez, cada panel es un **top 10 dibujado**,
y doce top 10 puestos en fila se comparan de un vistazo.

Esta forma tiene nombre: **small multiple**, o serie de pequeños múltiplos. Es
una de las formas más potentes que hay, y la idea es siempre la misma:

> **La afirmación no está en ningún panel. Está en la diferencia entre paneles.**

### Un detalle técnico que juega a favor

En el cuaderno 01 insistimos en que los conteos crudos no se pueden comparar
entre años de distinto tamaño. Acá estamos usando conteos crudos. ¿No es una
contradicción?

No, y la razón es una propiedad de esta forma: **cada nube se escala a su propio
máximo.** La palabra más frecuente de 2009 ocupa el tamaño máximo del panel de
2009, aunque en números absolutos sea mucho menor que la más frecuente de 2005.

O sea que la comparación entre paneles es de **composición y ranking**, no de
nivel absoluto. Y esa es justamente la pregunta que hicimos: no *cuánto* escribió
cada año, sino *de qué*.

Es un caso poco común: la forma visual esquiva sola un problema que en una tabla
había que resolver a mano. **No hay que dar por sentado que pasa; hay que
verificar qué normaliza cada herramienta.**

### Y algo que las nubes muestran sin que se lo hayamos pedido

Mirá qué palabra es la más grande en casi todos los paneles.

`si`. Y también `año`, `había`, `dos`, `cosa`.

Son palabras de relleno que sobrevivieron a nuestra lista de stopwords. Miremos
la lista.

In [ ]:
for palabra in ["sí", "si", "año", "había", "dos", "cosa", "entonces"]:
    esta = palabra in palabras_vacias
    print(" ", palabra, "-> ¿está en la lista de palabras vacías?", esta)

**`sí` con tilde está en la lista. `si` sin tilde, no.**

Son dos palabras distintas: una es afirmación, la otra es la conjunción
condicional. Y la conjunción condicional es de las palabras más frecuentes del
español. Quien armó la lista incluyó una y se olvidó de la otra.

Ese descuido estaba en todos los análisis que venimos haciendo desde el cuaderno
01. **Las doce nubes lo hicieron visible en dos segundos**, porque un problema
que se repite en los doce paneles se ve mucho más rápido que en una tabla de
frecuencias.

Arreglalo y volvé a correr la grilla.

In [ ]:
# Agregamos las palabras de relleno que la lista original no tenía.
# CADA UNA DE ESTAS ES UNA DECISIÓN: anotala y justificala.
palabras_agregadas = ["si", "año", "años", "había", "dos", "cosa", "vez",
                      "ahora", "entonces", "siempre", "después", "ser", "día"]

lista_ampliada = list(palabras_vacias)
for palabra in palabras_agregadas:
    lista_ampliada.append(palabra)

print("lista original:", len(palabras_vacias))
print("lista ampliada:", len(lista_ampliada))

In [ ]:
figura, paneles = plt.subplots(4, 3, figsize=(13.5, 14))
figura.patch.set_facecolor(SUPERFICIE)
lista_de_paneles = paneles.flatten()

for posicion in range(len(anios)):
    anio = anios[posicion]
    texto_del_anio = " ".join(palabras_por_anio[anio])

    nube_del_anio = WordCloud(
        width=640, height=420, background_color=SUPERFICIE,
        stopwords=set(lista_ampliada), max_words=10, max_font_size=100,
        min_font_size=26, random_state=42, prefer_horizontal=1.0,
        relative_scaling=0.75, margin=12, font_path=RUTA_FUENTE,
    )
    nube_del_anio.generate(texto_del_anio)

    puestos.clear()
    puesto = 0
    for palabra in nube_del_anio.words_:
        puestos[palabra] = puesto
        puesto = puesto + 1
    nube_del_anio.recolor(color_func=color_por_puesto)

    panel = lista_de_paneles[posicion]
    panel.imshow(nube_del_anio, interpolation="bilinear")
    panel.axis("off")
    panel.grid(False)
    panel.set_title(str(anio), fontsize=14, pad=10)

figura.suptitle("Con la lista de palabras vacías ampliada", fontsize=17, y=0.998)
figura.tight_layout(h_pad=2.2, w_pad=1.2)
plt.show()

Ahora se lee una biografía.

- **2004:** `alex`, `lucas` — los dos chicos de los diálogos.
- **2007:** `argentino`, `mundo`, `hombre` — son los años de escribir desde Barcelona.
- **2010 a 2012:** `revista`, `orsai`, `chiri`, `lectores`, `número` — Editorial Orsai.
- **2013:** `papelito`, `libro`.
- **2015:** `casa`, `hija`, `padre`.

Fijate en `casa`: **no la sacamos**, y ahora domina 2008, 2009, 2014 y 2015. En
la primera grilla estaba tapada por `si` y `año`. No era relleno: era un tema.

> **Ojo con lo que acabamos de hacer.** Agregamos trece palabras a la lista
> mirando el resultado y decidiendo que no nos gustaba.
>
> Eso es legítimo y lo hace todo el mundo, **pero es una intervención sobre los
> datos, no una limpieza técnica**. Si hubiéramos metido `casa` en esa lista,
> habríamos borrado un tema creyendo que sacábamos ruido.
>
> Cada palabra que sacás tiene que estar anotada y alguien tiene que poder
> discutírtela.

## 3. Elegir la forma

Antes de dibujar hay que saber qué relación queremos mostrar. La forma sale de
ahí, no del catálogo de gráficos.

| lo que quiero mostrar | forma |
|---|---|
| comparar categorías entre sí | barras |
| un cambio a lo largo del tiempo | línea |
| una distribución | histograma o puntos |
| una sola cifra que importa | el número, sin gráfico |

Nada más. Con esas cuatro se resuelve casi todo lo que vas a necesitar.

## 4. Los mismos datos, dos historias

Volvamos a la palabra `padre`, del cuaderno 01.

**La afirmación primero, el gráfico después.** Esta es la regla del cuaderno: no
dibujamos nada hasta haber escrito qué queremos afirmar.

In [ ]:
def frecuencias_de(palabra_buscada):
    # Devuelve, para cada año, las veces que aparece la palabra y su
    # frecuencia por cada 10.000 palabras del año.
    filas = []
    for anio in anios:
        veces = palabras_por_anio[anio].count(palabra_buscada)
        por_diez_mil = 10000 * veces / total_por_anio[anio]
        fila = {"anio": anio, "veces": veces, "por_10k": round(por_diez_mil, 2)}
        filas.append(fila)
    return pd.DataFrame(filas)


padre = frecuencias_de("padre")
padre

Ahora los dos gráficos, uno al lado del otro. **Misma palabra, mismo corpus,
misma forma. Lo único que cambia es qué se cuenta.**

In [ ]:
figura, (izquierda, derecha) = plt.subplots(1, 2, figsize=(11, 4))

izquierda.plot(padre["anio"], padre["veces"], color=NEUTRO, linewidth=2, marker="o", markersize=6)
izquierda.set_title("Conteo crudo")
izquierda.set_ylabel("veces que aparece")
izquierda.set_ylim(bottom=0)

derecha.plot(padre["anio"], padre["por_10k"], color=VIOLETA, linewidth=2, marker="o", markersize=6)
derecha.set_title("Por cada 10.000 palabras")
derecha.set_ylabel("frecuencia relativa")
derecha.set_ylim(bottom=0)

plt.show()

En el izquierdo, 2007 y 2015 parecen muy distintos: 34 contra 23.

En el derecho quedan casi empatados, porque 2007 es un año largo y 2015 uno
corto.

> **Por qué van uno al lado del otro y no superpuestos.** Son dos escalas
> distintas —veces y veces-por-diez-mil—. Poner dos escalas en un mismo gráfico,
> con un eje a cada lado, es el error más común de la visualización de datos: el
> lector no tiene forma de saber cuál curva corresponde a cuál eje, y la posición
> relativa entre las dos curvas la elige quien dibuja.
>
> Cuando tengas dos medidas de escalas distintas: **dos gráficos**.

## 5. El eje que no empieza en cero

Otra manera de que la misma serie cuente historias distintas.

In [ ]:
figura, (izquierda, derecha) = plt.subplots(1, 2, figsize=(11, 4))

izquierda.bar(padre["anio"], padre["por_10k"], color=VIOLETA, width=0.6)
izquierda.set_title("Con el eje desde cero")
izquierda.set_ylim(bottom=0)

derecha.bar(padre["anio"], padre["por_10k"], color=VIOLETA, width=0.6)
derecha.set_title("Con el eje desde 4")
derecha.set_ylim(bottom=4)

plt.show()

En el derecho, 2012 desapareció y 2015 parece cinco veces más alto que 2006.
Los datos son idénticos.

**En un gráfico de barras el eje tiene que empezar en cero**, porque lo que el
ojo compara es el largo de la barra. Si el eje arranca en 4, el largo deja de ser
proporcional al valor.

En un gráfico de líneas la regla es menos estricta —ahí lo que se lee es la
forma, no el largo— pero recortar el eje sigue exagerando las diferencias, y hay
que decirlo.

## 6. El título es parte del argumento

Mismo gráfico, dos títulos.

In [ ]:
figura, (izquierda, derecha) = plt.subplots(1, 2, figsize=(11, 4))

izquierda.plot(padre["anio"], padre["por_10k"], color=VIOLETA, linewidth=2, marker="o", markersize=6)
izquierda.set_title("Frecuencia de 'padre' por año")
izquierda.set_ylim(bottom=0)

derecha.plot(padre["anio"], padre["por_10k"], color=VIOLETA, linewidth=2, marker="o", markersize=6)
derecha.set_title("En 2012 la palabra 'padre' casi desaparece")
derecha.set_ylim(bottom=0)

# Señalamos el punto del que habla el título.
# La flecha va del texto al dato; el texto se ubica donde no tape la línea.
valor_2012 = padre[padre["anio"] == 2012]["por_10k"].iloc[0]

derecha.annotate(
    "1,01 por 10.000",
    xy=(2012, valor_2012),
    xytext=(2007.4, 1.6),
    color=NEUTRO,
    arrowprops={"arrowstyle": "->", "color": NEUTRO, "linewidth": 1},
)

plt.show()

El de la izquierda es un **título-etiqueta**: dice qué se está mirando.

El de la derecha es un **título-tesis**: afirma algo, y la anotación dirige la
mirada al lugar donde eso se ve.

Los dos son legítimos. Pero el segundo **se hace cargo**: dice qué hay que ver, y
por lo tanto se puede discutir. El primero deja que el lector saque la conclusión
que quiera y después nadie se hace responsable.

> Cuando escribas un título-tesis, tenés que poder contestar: **¿la imagen
> sostiene esto?** Si el gráfico no muestra lo que el título afirma, el problema
> es el título.

## 7. Comparar categorías: las palabras de un año

Para comparar valores entre categorías —palabras, en este caso— la forma es
barras. Y como los nombres son largos, van **horizontales**.

Usamos el cálculo de palabras características del cuaderno 02.

In [ ]:
def caracteristicas_de(anio_elegido, cuantas):
    # Compara la frecuencia de cada palabra en un año contra la del resto.
    conteo_del_anio = Counter()
    for palabra in palabras_por_anio[anio_elegido]:
        if palabra in palabras_vacias:
            continue
        if len(palabra) < 3:
            continue
        conteo_del_anio[palabra] = conteo_del_anio[palabra] + 1

    conteo_del_resto = Counter()
    total_del_resto = 0
    for anio in anios:
        if anio == anio_elegido:
            continue
        for palabra in palabras_por_anio[anio]:
            if palabra in palabras_vacias:
                continue
            if len(palabra) < 3:
                continue
            conteo_del_resto[palabra] = conteo_del_resto[palabra] + 1
        total_del_resto = total_del_resto + total_por_anio[anio]

    filas = []
    for palabra in conteo_del_anio:
        veces = conteo_del_anio[palabra]
        if veces < 10:
            continue
        aca = 10000 * veces / total_por_anio[anio_elegido]
        alla = 10000 * conteo_del_resto[palabra] / total_del_resto
        fila = {"palabra": palabra, "cuantas_veces_mas": round(aca / (alla + 0.5), 1)}
        filas.append(fila)

    tabla = pd.DataFrame(filas)
    tabla = tabla.sort_values("cuantas_veces_mas", ascending=False)
    return tabla.head(cuantas).reset_index(drop=True)


palabras_2010 = caracteristicas_de(2010, 8)
palabras_2010

In [ ]:
# Las damos vuelta para que la más alta quede arriba en el gráfico.
para_dibujar = palabras_2010.iloc[::-1]

plt.figure(figsize=(8, 4))
plt.barh(para_dibujar["palabra"], para_dibujar["cuantas_veces_mas"], color=VIOLETA, height=0.6)
plt.title("En 2010 Casciari escribe sobre librerías")
plt.xlabel("cuántas veces más frecuente que en el resto del corpus")
plt.grid(axis="y", visible=False)
plt.show()

Fijate lo que **no** hicimos: poner el número arriba de cada barra. El eje ya
dice la escala, y un número en cada barra es ruido.

Etiquetar directamente sirve cuando querés destacar **uno** —el mayor, el que te
interesa—, no todos.

## 8. El gráfico que no sostiene su afirmación

Cerramos con el caso más importante, porque es el que se ve menos venir.

Hay una medida clásica de riqueza léxica: el **TTR**, *type-token ratio*. Es
cuántas palabras distintas usa un texto dividido cuántas palabras usa en total.
Cuanto más alto, más variado sería el vocabulario.

Calculémoslo por año.

In [ ]:
filas = []
for anio in anios:
    palabras = palabras_por_anio[anio]
    distintas = len(set(palabras))
    fila = {
        "anio": anio,
        "palabras_del_anio": len(palabras),
        "palabras_distintas": distintas,
        "ttr": round(distintas / len(palabras), 3),
    }
    filas.append(fila)

riqueza = pd.DataFrame(filas)
riqueza

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(riqueza["anio"], riqueza["ttr"], color=VIOLETA, linewidth=2, marker="o", markersize=6)
plt.title("En 2009 Casciari usó el vocabulario más variado")
plt.ylabel("TTR")
plt.ylim(bottom=0)
plt.show()

El título afirma algo y el gráfico parece sostenerlo: 2009 es el pico.

**Ahora la pregunta de siempre: ¿la imagen sostiene lo que la frase afirma?**

Miremos el TTR contra el tamaño de cada año.

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.scatter(riqueza["palabras_del_anio"], riqueza["ttr"], color=VIOLETA, s=60)

# Etiquetamos solo los dos extremos, no los doce.
for posicion in [0, len(riqueza) - 1]:
    fila_extrema = riqueza.sort_values("palabras_del_anio").iloc[posicion]
    plt.annotate(
        str(int(fila_extrema["anio"])),
        xy=(fila_extrema["palabras_del_anio"], fila_extrema["ttr"]),
        xytext=(6, 4),
        textcoords="offset points",
        color=NEUTRO,
    )

plt.title("Cuanto más corto el año, más alto el TTR")
plt.xlabel("palabras del año")
plt.ylabel("TTR")
plt.show()

In [ ]:
# math: para la raíz cuadrada de la correlación.
import math

tamanos = list(riqueza["palabras_del_anio"])
valores_ttr = list(riqueza["ttr"])

media_tamano = sum(tamanos) / len(tamanos)
media_ttr = sum(valores_ttr) / len(valores_ttr)

arriba = 0
for posicion in range(len(tamanos)):
    arriba = arriba + (tamanos[posicion] - media_tamano) * (valores_ttr[posicion] - media_ttr)

suma_tamanos = 0
suma_ttr = 0
for posicion in range(len(tamanos)):
    suma_tamanos = suma_tamanos + (tamanos[posicion] - media_tamano) ** 2
    suma_ttr = suma_ttr + (valores_ttr[posicion] - media_ttr) ** 2

correlacion = arriba / math.sqrt(suma_tamanos * suma_ttr)

print("correlación entre el tamaño del año y su TTR:", round(correlacion, 3))

**−0,88.**

Casi toda la variación del TTR se explica por el tamaño del año. No es un
hallazgo sobre la escritura de Casciari: es una propiedad aritmética de la
medida.

La razón es simple. Un texto corto tiene pocas oportunidades de repetir palabras.
A medida que un texto crece, las palabras nuevas escasean —ya usaste casi todas
las comunes— pero las repeticiones siguen acumulándose. El cociente baja siempre.

2009 no tiene el vocabulario más variado. **2009 es el archivo más chico.**

> El primer gráfico —el del pico en 2009— era correcto en todos sus números y
> falso en su afirmación. Ninguna validación técnica lo habría detectado.
>
> Lo detectamos porque preguntamos **si la imagen sostiene lo que dice el
> título**, y fuimos a buscar qué otra cosa podía explicar la forma.

## 9. La auditoría

Antes de mostrarle un gráfico a alguien, seis preguntas.

1. **¿Qué afirma el título?** Si no afirma nada, ¿por qué estoy mostrando esto?
2. **¿La imagen sostiene esa afirmación?**
3. **¿Los valores están normalizados?** ¿Estoy comparando la cosa o el tamaño de
   la muestra?
4. **¿El eje empieza en cero?** Si no, ¿lo dije?
5. **¿Hay otra explicación para la forma que veo?** Como el tamaño del archivo en
   el TTR.
6. **¿Qué dejé afuera?** Un top 10 es una decisión: hay un puesto 11.

Las seis se pueden contestar en un minuto. Los errores que evitan cuestan meses.

> ### Perilla
>
> Elegí una palabra que te interese —`madre`, `libro`, `fútbol`, `hija`,
> `internet`— y armá **un** gráfico con un título-tesis.
>
> Después pasá las seis preguntas de la auditoría. Si alguna te deja incómodo,
> cambiá el gráfico o cambiá el título. No dejes el título si la imagen no lo
> sostiene.

In [ ]:
palabra_elegida = "libro"

tabla_elegida = frecuencias_de(palabra_elegida)

plt.figure(figsize=(8, 4))
plt.plot(tabla_elegida["anio"], tabla_elegida["por_10k"], color=VIOLETA, linewidth=2, marker="o", markersize=6)
plt.title("ESCRIBÍ ACÁ TU TÍTULO-TESIS")
plt.ylabel("por cada 10.000 palabras")
plt.ylim(bottom=0)
plt.show()

tabla_elegida

## Lo que hiciste hoy

- Viste que una nube sola no sostiene ninguna afirmación, y que doce nubes
  comparables sí, porque la afirmación está en la diferencia entre paneles.
- Elegiste la forma del gráfico a partir de la relación que querías mostrar.
- Viste los mismos datos contar dos historias, por normalizar o no.
- Viste una tercera historia salir del mismo dato, moviendo el eje.
- Distinguiste título-etiqueta de título-tesis.
- Encontraste un gráfico con todos los números correctos y la afirmación falsa.

## Cierre de la unidad

Empezamos con doce archivos que no se podían abrir. Terminamos con una serie de
afirmaciones sobre doce años de escritura, cada una con su evidencia y sus
decisiones a la vista.

En el medio, en cada paso, hubo una elección: qué considerar una palabra, qué
sacar, con qué comparar, qué unidad usar, qué mostrar y cómo titularlo.

> **Ninguna de esas elecciones era obligatoria, y todas cambian el resultado.**
>
> Eso no invalida el análisis. Lo que lo invalidaría es no poder decir cuáles
> fueron.

## Lo que viene

Todo esto lo hicimos contando palabras sueltas. La computadora nunca supo que
`padre` es un sustantivo, que `escribía` viene de `escribir`, ni que `Mercedes`
puede ser una ciudad o una persona.

Eso es lo que agrega spaCy, y es la unidad que sigue.